In [50]:
import os
import json
import requests
from dotenv import load_dotenv

# .env 파일로부터 환경변수 로드
load_dotenv() 
KEY = os.environ.get("KOREAN_DICT_KEY")
print(KEY[:4]+"****") 

74B7****


In [ ]:
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY, 
        "q": q, 
        "req_type": "json", 
        "num": num, 
        "start": start, 
        "type1": "word" 
    }
    
    
    response = requests.get(url, params=params, timeout=10) 
    
    response.raise_for_status() 
    
    return response.json()
#우리말 샘의 오픈 api를 활용하여 특정한 검색어의 사전데이터를 가져오는 함수이다. requests.get으로 인증키와 검색조건을 조정하고 일정시간 내에 응답이 없으면 예외가 발생하도록 timeout을 설정하였다. 이휴 오류검사를 거쳐 json 형태의 딕셔너리로 반환한다.

In [ ]:
import json 
data=search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])
#김치의 json형태 검색 결과를 사람이 보기 편하도록 변환 후 앞 400자를 출력한다. ensure_ascii=False가 없다면 파이썬은 한글을 아스키 범위 내의 유니코드 이스케이프 시퀀스 형태로 강제 변환하여 사람이 읽을수 없게 출력한다.

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


In [ ]:
channel = data.get("channel", {})
total = channel.get("total", 0)
items = channel.get("item", [])
n = len(items)


print(f"총 {total}건, 이 페이지 {n}건")
for item in items[:5]:
    word = item.get("word")
    
   
    pos = item.get("pos", "품사 없음")
    if not pos: 
        pos = "품사 없음"
        
  
    senses = item.get("sense", [])
    
   
    if isinstance(senses, list) and len(senses) > 0:
        first_sense = senses[0] 
        definition = first_sense.get("definition", "뜻풀이 없음")
    elif isinstance(senses, dict): 
        definition = senses.get("definition", "뜻풀이 없음")
    else:
        definition = "뜻풀이 없음"
        
    
    short_def = definition[:40]
    
    
    print(f"{word} ({pos}) -> {short_def}")
    #이 코드는 API 응답 데이터에서 전체 검색 결과 수와 현재 페이지의 항목 수를 추출하여 출력한다. 이어서 검색된 단어 중 상위 5개를 순회하며 표제어, 품사, 그리고 첫 번째 뜻풀이의 앞 40자를  지정된 형식으로 화면에 표시한다.

총 328건, 이 페이지 10건
김치 (품사 없음) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


In [ ]:
import time
words: list[str] = ["김치","라면","만두","김밥","국수","떡볶이","불고기","비빔밥"]
for q in words:
    data = search_word(q)
    total = data.get("channel", {}).get("total", 0)
    print(f"{q}: {total}건")
    time.sleep(0.3)

#음식 검색어 리스트를 돌면서 search_word 함수를 반복 호출해 전체 결과 수를 포맷에 맞게 출력하는 코드이다. 호출 사이에 0.3초씩 쉬어서 과부하를 방지한다.

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


In [62]:
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥"
]

all_items = []

for q in words:
    data = search_word(q)
    items = data.get("channel", {}).get("item", [])
    all_items.extend(items)
    time.sleep(0.3)


pos_list = []
for item in all_items:
    pos = item.get("pos")
    if not pos:
        senses = item.get("sense", [])
        if isinstance(senses, list) and len(senses) > 0:
            pos = senses[0].get("pos")
        elif isinstance(senses, dict):
            pos = senses.get("pos")
            
    final_pos = pos or "(미상)"
    pos_list.append(final_pos)

print("[품사 빈도 상위 3개]")
pos_counts = Counter(pos_list)
for pos, count in pos_counts.most_common(3):
    print(f"{pos}: {count}회")
#8개 단어의 모든 검색 결과 항목을 하나의 리스트로 합친 뒤, 각 항목의 품사를 가져와 가장 많이 등장한 상위 3개를 출력하는 코드이다. 품사 필드가 비어있을 때는 or을 활용해 예외 처리를 하고 Counter의 most_common(3) top3빈도를 계산했다.

[품사 빈도 상위 3개]
명사: 60회
(미상): 19회
어미: 1회
